In [1]:
import pandas as pd

books = pd.read_csv("books_with_predictions.csv")

In [2]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = "cpu")
classifier("I love this!")

C:\Users\chake\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528688922524452},
  {'label': 'neutral', 'score': 0.005764589179307222},
  {'label': 'anger', 'score': 0.004419790115207434},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.001611991785466671},
  {'label': 'fear', 'score': 0.0004138524236623198}]]

In [3]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)

In [10]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import time

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]

def process_all_books_optimized(books_df, classifier, sentence_batch_size=32):
    
    results = []
    errors = []
    
    for idx, row in tqdm(books_df.iterrows(), total=len(books_df), desc="Analyzing emotions"):
        isbn = row["isbn13"]
        description = row.get("description", "")
        
        # Split into sentences
        sentences = [s.strip() for s in str(description).split(".") if s.strip()]
        
        if not sentences:
            continue
        
        try:
            # Process sentences in batches for speed
            emotion_max_scores = {label: 0.0 for label in emotion_labels}
            
            # Process in batches of 32 sentences at a time
            for i in range(0, len(sentences), sentence_batch_size):
                batch_sentences = sentences[i:i+sentence_batch_size]
                predictions = classifier(batch_sentences)
                
                # Update max scores
                for sentence_pred in predictions:
                    pred_dict = {p["label"]: p["score"] for p in sentence_pred}
                    
                    for label in emotion_labels:
                        if label in pred_dict:
                            emotion_max_scores[label] = max(
                                emotion_max_scores[label], 
                                pred_dict[label]
                            )
            
            # Store result
            result = {"isbn13": isbn}
            result.update(emotion_max_scores)
            results.append(result)
            
        except Exception as e:
            errors.append({"isbn13": isbn, "error": str(e)})
            continue

    # Create DataFrame
    emotion_df = pd.DataFrame(results)
    
    if len(emotion_df) > 0:
        emotion_df["dominant_emotion"] = emotion_df[emotion_labels].idxmax(axis=1)
        emotion_df["max_score"] = emotion_df[emotion_labels].max(axis=1)
    
    return emotion_df, errors

# Process entire dataset
emotion_df, errors = process_all_books_optimized(books, classifier, sentence_batch_size=32)




Analyzing emotions: 100%|██████████| 5197/5197 [14:01<00:00,  6.18it/s] 


In [17]:

emotion_df.to_csv("books_emotion_scores.csv", index=False)



In [20]:
books = pd.merge(books, emotion_df, on = "isbn13")

In [22]:
books.to_csv("books_with_emotions.csv", index = False)